# **ARTIFICIAL INTELLIGENCE CAPSTONE PROJECT – PRAICP-1002 : Traffic Sign Detection & Classification**

**PROJECT TITLE:** PRAICP-1002: TrafSignDetc – Traffic Sign Recognition

**TEAM CODE:**  PTID-AIE-JUL-26-11142

**PROJECT CODE:** PRAICP-1002

**PROJECT TYPE:** Artificial Intelligence Capstone Project

**DATASET SOURCE:** DataMites Capstone Project Dataset

**DATASET NAME:** TrafSignDetc (directories of traffic sign images, one directory per sign)

**DATASET LINK:** https://d3ilbtxij3aepc.cloudfront.net/projects/AI-Capstone-Projects/PRAICP-1002-TrafSignDetc.zip

**SUBMITTED BY:** Sameeksha

## **1. Problem Statement & Project Objective**

**1.  Problem Statement**

Traffic signs tell drivers what they must do, what they must not do and what dangers lie ahead. Missing or misreading a sign (a stop sign, a speed limit or a warning) is a common cause of road accidents. Driver-assistance systems and self-driving vehicles therefore need to *recognise traffic signs automatically* from camera images. The objective of this project is to develop a Convolutional Neural Network (CNN) based machine learning model that can look at a photograph of a traffic sign from the TrafSignDetc dataset and automatically predict which traffic sign it is, helping to improve road safety.

**2. Project Objective**

The DataMites project brief defines two goals, both of which are covered in this notebook:

1. **Image preprocessing** – convert raw, variable-resolution traffic sign photographs into a clean, consistent numerical format (resize, normalise, augment) that a neural network can learn from.
2. **ML model to predict the Traffic Sign** – build, tune and evaluate a CNN that classifies each image into its traffic sign class.

To achieve this, the notebook will:

1. Load and explore the dataset (classes, class balance, image sizes, sample images) – in a **memory-safe way**, because the full dataset is very large.
2. Preprocess the images (resize to a fixed size, normalise pixel values) and split them into Training / Validation / Test sets.
3. Apply data augmentation (small rotations, shifts, zooms and contrast changes) to improve generalisation.
4. Build a baseline CNN , then improve it with hyperparameter tuning (Keras Tuner) to obtain a final CNN.
5. Evaluate the models with Accuracy, Precision, Recall, F1-score, a Confusion Matrix and a Classification Report.
6. Predict the traffic sign for a brand-new image and save the trained model.

## **2. Import Python Libraries**

The required Python libraries are imported to perform dataset downloading, image loading and preprocessing, visualisation, CNN model development, hyperparameter tuning and performance evaluation. TensorFlow/Keras is used to build and train the CNN models, Keras Tuner is used to search for the best hyperparameters, and Scikit-learn provides the data-splitting utilities and evaluation metrics.

In [ ]:
# Keras Tuner (hyperparameter tuning), tensorboard (required by Keras Tuner) and tqdm (progress bars) - needed on Google Colab; harmless if already installed
!pip install -q keras-tuner tensorboard tqdm

In [ ]:
# Core Libraries
import os
import re
import json
import gc
import random
import zipfile
import posixpath
import textwrap
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# Downloading the dataset
import requests

# Scikit-learn utilities
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, precision_recall_fscore_support)

# Deep Learning Libraries
import tensorflow as tf
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Rescaling,
                                     RandomRotation, RandomTranslation, RandomZoom, RandomContrast)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Hyperparameter tuning
import keras_tuner as kt

# Reproducibility: fix the random seeds so results are repeatable
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_style('whitegrid')

print('TensorFlow version :', tf.__version__)
print('Keras Tuner version:', kt.__version__)
print('GPU available      :', len(tf.config.list_physical_devices('GPU')) > 0)

## **3. Upload the Dataset And Domain Analysis**

**1. Upload the Dataset**

The dataset is provided by DataMites as a ZIP file. It contains one directory per traffic sign, and each directory holds the photographs of that sign. The directory name is therefore the label (class) of every image inside it.

**The full dataset is very large** (tens of thousands of images). Extracting everything and loading it all into memory as arrays would exhaust the RAM of a normal laptop or a free Google Colab session. To avoid this, the code below follows a **memory-safe strategy**:

1. The ZIP is downloaded once (skipped automatically if it is already present).
2. The ZIP is **read without extracting it** to find the class directories and count the images in each class (this is the *full dataset* picture used in the EDA).
3. Only a **fixed number of randomly chosen images per class** (`MAX_IMAGES_PER_CLASS`) is extracted to disk and used for the project. Choosing images *per class* keeps every sign represented.
4. Images are stored as compact `uint8` arrays at a small size (see Section 6), so the whole working set fits comfortably in memory.

Set `MAX_IMAGES_PER_CLASS = None` to use the complete dataset if your machine has enough RAM, or lower it (e.g. `100`) for a very quick dry-run.

**2. Domain Analysis**

Traffic signs are designed to be recognised quickly by humans, so they follow strict visual rules: the **shape** of the sign (circle, triangle, octagon, diamond), its **border and background colour** (red for prohibition/warning, blue for mandatory instructions, yellow for priority) and the **pictogram or number** inside it. These are spatial visual patterns, which makes this a natural candidate for a CNN – it learns edges, shapes, colours and symbols directly from pixels instead of relying on hand-crafted features.

Real-world traffic sign photographs are, however, **not clean**: they are taken from moving vehicles, so images can be blurred, very small, too dark or too bright, partly hidden, or photographed from an angle. Several signs are also very similar to each other (for example the different *speed limit* signs differ only in the number inside the circle), which makes some classes harder to separate than others. A further point is that **left/right-meaning signs (e.g. "Turn left ahead" vs "Turn right ahead") are mirror images of each other**, so mirroring an image must never be used as data augmentation here.

In [ ]:
# Download the dataset ZIP (skipped automatically if already present)
DATASET_URL = 'https://d3ilbtxij3aepc.cloudfront.net/projects/AI-Capstone-Projects/PRAICP-1002-TrafSignDetc.zip'
ZIP_PATH    = 'PRAICP-1002-TrafSignDetc.zip'

# ---- SIZE CONTROL -------------------------------------------------------------------------------
# Images used PER CLASS. The full dataset is too large to load into memory, so a random subset is used.
# 400 per class is a good balance of accuracy and speed. Use None for the full dataset, or e.g. 100 for a quick dry-run.
MAX_IMAGES_PER_CLASS = 400
# Set True to delete the big ZIP file after the needed images have been extracted (saves disk space)
DELETE_ZIP_AFTER_EXTRACT = False
# -------------------------------------------------------------------------------------------------

EXTRACT_DIR = f'TrafSign_data_{MAX_IMAGES_PER_CLASS if MAX_IMAGES_PER_CLASS else "all"}'

def download_file(url, dest_path, chunk_size=1 << 20):
    """Stream-download a file with a simple progress printout (the file is never held fully in memory)."""
    tmp_path = dest_path + '.part'
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        downloaded = 0
        with open(tmp_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total:
                        print(f'\rDownloading... {downloaded/1e6:,.1f} MB / {total/1e6:,.1f} MB '
                              f'({downloaded/total*100:.0f}%)', end='')
    os.replace(tmp_path, dest_path)          # only a fully downloaded file gets the final name
    print('\nDownload complete.')

if not os.path.exists(ZIP_PATH) and not os.path.isdir(EXTRACT_DIR):
    download_file(DATASET_URL, ZIP_PATH)
elif os.path.exists(ZIP_PATH):
    print('ZIP file already present, skipping download.')

In [ ]:
# Read the ZIP index WITHOUT extracting: find the folder that holds the class (sign) directories and count images per class
IMG_EXT = ('.jpg', '.jpeg', '.png', '.ppm', '.bmp')

def natural_key(s):
    """Sort '2' before '10' (plain alphabetical sorting would give 0, 1, 10, 11, ... 2)."""
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', s)]

def index_zip(zip_path):
    """Return (class_root, {class_name: [zip members]}) for the folder that contains the most class directories."""
    with zipfile.ZipFile(zip_path) as zf:
        members = [m for m in zf.namelist()
                   if m.lower().endswith(IMG_EXT) and not m.startswith('__MACOSX') and '/.' not in m]
    by_parent = {}
    for m in members:
        by_parent.setdefault(posixpath.dirname(m), []).append(m)       # directory -> its images
    by_grand = {}
    for parent in by_parent:
        by_grand.setdefault(posixpath.dirname(parent), []).append(parent)   # directory -> its image-holding sub-directories
    class_root = max(by_grand, key=lambda g: len(by_grand[g]))         # e.g. 'Train' - ignores flat folders such as a test set
    class_members = {posixpath.basename(p): by_parent[p] for p in by_grand[class_root]}
    return class_root, class_members

if os.path.exists(ZIP_PATH):
    CLASS_ROOT, zip_members = index_zip(ZIP_PATH)
    full_counts = pd.Series({c: len(v) for c, v in zip_members.items()})
    full_counts = full_counts.reindex(sorted(full_counts.index, key=natural_key))
    print('Class folder inside the ZIP :', repr(CLASS_ROOT))
    print('Classes found               :', len(full_counts))
    print('Images in the FULL dataset  :', f'{full_counts.sum():,}')
else:
    raise FileNotFoundError('ZIP not found. Check the download above, or place the ZIP next to this notebook.')

In [ ]:
# Extract ONLY the chosen images (a random MAX_IMAGES_PER_CLASS from every class) - not the whole ZIP
if not os.path.isdir(EXTRACT_DIR):
    rng = random.Random(SEED)
    selected = []
    for cls, files in zip_members.items():
        files = sorted(files)
        selected += files if MAX_IMAGES_PER_CLASS is None or len(files) <= MAX_IMAGES_PER_CLASS \
                    else rng.sample(files, MAX_IMAGES_PER_CLASS)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        for m in tqdm(selected, desc='Extracting selected images'):
            zf.extract(m, EXTRACT_DIR)
    print(f'Extracted {len(selected):,} of {full_counts.sum():,} images into "{EXTRACT_DIR}".')
else:
    print(f'"{EXTRACT_DIR}" already exists, skipping extraction.')

if DELETE_ZIP_AFTER_EXTRACT and os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)
    print('ZIP file deleted to save disk space.')

DATASET_PATH = os.path.join(EXTRACT_DIR, CLASS_ROOT)
CLASS_NAMES  = sorted((d for d in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, d))),
                      key=natural_key)
NUM_CLASSES  = len(CLASS_NAMES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

print('Dataset folder   :', DATASET_PATH)
print('Number of classes:', NUM_CLASSES)
print('Classes          :', CLASS_NAMES)

In [ ]:
# Human-readable sign names. The folders are numbered 0-42, which matches the standard 43-class traffic sign numbering
# (German Traffic Sign Recognition Benchmark). If your folders are named differently, the folder names are used instead.
GTSRB_NAMES = {
    0: 'Speed limit (20km/h)', 1: 'Speed limit (30km/h)', 2: 'Speed limit (50km/h)', 3: 'Speed limit (60km/h)',
    4: 'Speed limit (70km/h)', 5: 'Speed limit (80km/h)', 6: 'End of speed limit (80km/h)', 7: 'Speed limit (100km/h)',
    8: 'Speed limit (120km/h)', 9: 'No passing', 10: 'No passing for vehicles over 3.5 tons',
    11: 'Right-of-way at next intersection', 12: 'Priority road', 13: 'Yield', 14: 'Stop', 15: 'No vehicles',
    16: 'Vehicles over 3.5 tons prohibited', 17: 'No entry', 18: 'General caution', 19: 'Dangerous curve to the left',
    20: 'Dangerous curve to the right', 21: 'Double curve', 22: 'Bumpy road', 23: 'Slippery road',
    24: 'Road narrows on the right', 25: 'Road work', 26: 'Traffic signals', 27: 'Pedestrians', 28: 'Children crossing',
    29: 'Bicycles crossing', 30: 'Beware of ice/snow', 31: 'Wild animals crossing',
    32: 'End of all speed and passing limits', 33: 'Turn right ahead', 34: 'Turn left ahead', 35: 'Ahead only',
    36: 'Go straight or right', 37: 'Go straight or left', 38: 'Keep right', 39: 'Keep left',
    40: 'Roundabout mandatory', 41: 'End of no passing', 42: 'End of no passing by vehicles over 3.5 tons'}

use_names = all(c.isdigit() and int(c) in GTSRB_NAMES for c in CLASS_NAMES)
SIGN_NAMES = {c: (GTSRB_NAMES[int(c)] if use_names else c) for c in CLASS_NAMES}   # class folder -> readable name

def label_of(class_index, width=22):
    """Readable, wrapped label such as '14: Stop' for plot titles."""
    c = CLASS_NAMES[class_index]
    return textwrap.fill(f'{c}: {SIGN_NAMES[c]}' if use_names else c, width)

print('Readable sign names available:', use_names)
if use_names:
    print('Example ->', CLASS_NAMES[14] if NUM_CLASSES > 14 else CLASS_NAMES[0], ':', SIGN_NAMES[CLASS_NAMES[14] if NUM_CLASSES > 14 else CLASS_NAMES[0]])

## **4. Basic Checks**

Basic checks are performed to understand the size, quality and balance of the dataset before exploratory analysis and model development. A table with one row per image is created (its class, file path, width, height and colour mode), every file is opened to make sure it is **not corrupt**, and the number of images per class is counted to see whether the dataset is balanced. The class balance of the **full** dataset (read from the ZIP index) and of the **working subset** actually used are both reported.

In [ ]:
# Build a table with one row per (extracted) image
records = []
for cls in CLASS_NAMES:
    folder = os.path.join(DATASET_PATH, cls)
    for fname in sorted(os.listdir(folder)):
        if fname.lower().endswith(IMG_EXT):
            records.append({'class': cls, 'file': fname, 'path': os.path.join(folder, fname)})

df = pd.DataFrame(records)
print('Total number of images in the working set:', len(df))
df.head()

In [ ]:
# Check every image: is it readable (not corrupt)? What are its width, height and colour mode?
widths, heights, modes, corrupt_files = [], [], [], []

for p in tqdm(df['path'], desc='Checking images'):
    try:
        with Image.open(p) as img:
            w, h = img.size
            mode = img.mode
            img.verify()                  # raises an exception for corrupt / truncated files
        widths.append(w); heights.append(h); modes.append(mode)
    except Exception:
        widths.append(None); heights.append(None); modes.append(None)
        corrupt_files.append(p)

df['width'], df['height'], df['mode'] = widths, heights, modes

print('Number of corrupt/unreadable images found:', len(corrupt_files))
if corrupt_files:
    print(corrupt_files[:10])
    df = df[~df['path'].isin(corrupt_files)].reset_index(drop=True)    # drop them so they cannot break training
    print('Corrupt files removed. Remaining images:', len(df))

In [ ]:
# Images per class and overall class balance - FULL dataset (from the ZIP index) and the WORKING subset (used here)
class_counts = df['class'].value_counts().reindex(CLASS_NAMES)

print('FULL DATASET (inside the ZIP)')
print('  Total images           :', f'{full_counts.sum():,}')
print('  Smallest class         :', full_counts.idxmin(), '->', full_counts.min(), 'images')
print('  Largest class          :', full_counts.idxmax(), '->', full_counts.max(), 'images')
print('  Imbalance ratio (max/min): {:.2f} : 1'.format(full_counts.max() / full_counts.min()))
print()
print('WORKING SUBSET (used in this notebook, max', MAX_IMAGES_PER_CLASS, 'per class)')
print('  Total images           :', f'{len(df):,}')
print('  Smallest class         :', class_counts.idxmin(), '->', class_counts.min(), 'images')
print('  Largest class          :', class_counts.idxmax(), '->', class_counts.max(), 'images')
print('  Imbalance ratio (max/min): {:.2f} : 1'.format(class_counts.max() / class_counts.min()))
print()
print('File formats             :', sorted(set(os.path.splitext(f)[1].lower() for f in df['file'])))
print('Colour modes             :', df['mode'].value_counts().to_dict())

In [ ]:
# Image resolutions present in the dataset (traffic sign photos come in many different sizes)
print('Width  (pixels): min =', int(df['width'].min()),  ' median =', int(df['width'].median()),  ' max =', int(df['width'].max()))
print('Height (pixels): min =', int(df['height'].min()), ' median =', int(df['height'].median()), ' max =', int(df['height'].max()))
print('Number of different resolutions:', df.groupby(['width', 'height']).ngroups)

size_table = (df.groupby(['width', 'height']).size()
                .reset_index(name='Number of images')
                .sort_values('Number of images', ascending=False)
                .head(10).reset_index(drop=True))
size_table

## **5. Exploratory Data Analysis (EDA) — Data Analysis Report**

Exploratory Data Analysis is performed to visually understand the dataset before building the CNN. This includes the class distribution, the resolutions of the original images, one sample image from every traffic sign, and several samples from a single sign to see how much variation exists *within* a class (different lighting, blur, angles and sizes).

In [ ]:
# Class distribution of the FULL dataset
plt.figure(figsize=(15, 5))
sns.barplot(x=full_counts.index, y=full_counts.values, palette='mako')
plt.axhline(full_counts.mean(), color='red', linestyle='--', label=f'Mean = {full_counts.mean():.0f} images/class')
plt.title('Number of Images per Traffic Sign (Class) - Full Dataset')
plt.xlabel('Traffic Sign (Class)')
plt.ylabel('Number of Images')
plt.xticks(rotation=90)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Class distribution of the WORKING subset (after capping the images per class)
plt.figure(figsize=(15, 5))
sns.barplot(x=class_counts.index, y=class_counts.values, palette='rocket')
plt.axhline(class_counts.mean(), color='red', linestyle='--', label=f'Mean = {class_counts.mean():.0f} images/class')
plt.title(f'Number of Images per Traffic Sign (Class) - Working Subset (max {MAX_IMAGES_PER_CLASS} per class)')
plt.xlabel('Traffic Sign (Class)')
plt.ylabel('Number of Images')
plt.xticks(rotation=90)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of the original image resolutions
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].hist(df['width'], bins=40, color='steelblue');   axes[0].set_title('Original Image Width (pixels)')
axes[0].set_xlabel('Width');  axes[0].set_ylabel('Number of Images')
axes[1].hist(df['height'], bins=40, color='darkorange'); axes[1].set_title('Original Image Height (pixels)')
axes[1].set_xlabel('Height'); axes[1].set_ylabel('Number of Images')
plt.tight_layout()
plt.show()

In [ ]:
# One sample image from every traffic sign
n_cols = 9
n_rows = int(np.ceil(NUM_CLASSES / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.3 * n_cols, 2.6 * n_rows))
for ax, (i, cls) in zip(axes.ravel(), enumerate(CLASS_NAMES)):
    sample_path = df[df['class'] == cls].sample(1, random_state=SEED)['path'].values[0]
    ax.imshow(Image.open(sample_path))
    ax.set_title(label_of(i, 16), fontsize=7)
    ax.axis('off')
for ax in axes.ravel()[NUM_CLASSES:]:
    ax.axis('off')
plt.suptitle('One Sample Image from Every Traffic Sign', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Variation WITHIN a class: 10 random images of the same sign (different lighting, blur, size and angle)
def show_class_variation(class_index, n=10):
    cls = CLASS_NAMES[class_index]
    files = df[df['class'] == cls].sample(n, random_state=SEED)['path'].values
    fig, axes = plt.subplots(2, n // 2, figsize=(2.6 * (n // 2), 5.6))
    for ax, p in zip(axes.ravel(), files):
        ax.imshow(Image.open(p))
        ax.axis('off')
    plt.suptitle(f'Variation within sign {label_of(class_index, 60)}', fontsize=15)
    plt.tight_layout()
    plt.show()

show_class_variation(0)
show_class_variation(NUM_CLASSES // 2)

In [ ]:
# EDA findings generated automatically from THIS run (nothing is hard-coded)
print('EDA FINDINGS')
print('-' * 70)
print(f'1. Classes                : {NUM_CLASSES} traffic sign classes')
print(f'2. Full dataset size      : {full_counts.sum():,} images  (working subset used: {len(df):,})')
print(f'3. Corrupt images found   : {len(corrupt_files)}')
print(f'4. Class imbalance (full) : {full_counts.max() / full_counts.min():.1f} : 1  '
      f'(smallest class {full_counts.idxmin()} = {full_counts.min()}, largest class {full_counts.idxmax()} = {full_counts.max()})')
print(f'5. Image size range       : {int(df.width.min())}x{int(df.height.min())} up to {int(df.width.max())}x{int(df.height.max())} pixels '
      f'({df.groupby(["width", "height"]).ngroups} different resolutions)')
print(f'6. Colour modes           : {df["mode"].value_counts().to_dict()}')

###**EDA Summary (Conclusion):**

1.The dataset contains one directory per traffic sign, and the directory name is the label of every image inside it. The exact number of classes, the total number of images and the corrupt-file check result are printed in the *EDA findings* cell above.

2.The classes are **not equally sized**: some signs have many times more images than others (the imbalance ratio of the full dataset is printed above). This is addressed later by capping the images per class, using **stratified splitting** (so every split keeps the same class proportions) and **class weights** during training.

3.The images come in **many different resolutions**, and many of them are small. A CNN needs a fixed input size, which confirms the need for the resizing step in the preprocessing section.

4.The sample images show that signs are photographed from vehicles: images within the same class vary in **brightness, sharpness, size, viewing angle and background**. This supports the use of **data augmentation** (small rotations, shifts, zoom and contrast changes) to help the model generalise.

5.Some signs are visually very close to each other (for example the different speed-limit signs, which differ only by the number inside the circle). These are expected to be the hardest to separate and will be examined later in the confusion matrix.

## **6. Image Preprocessing**

Image preprocessing is the first of the two goals in the project brief. A CNN cannot accept images of different sizes and learns best when its inputs are on a small, consistent numerical scale. Two preprocessing steps are therefore applied to **every** image:

**1. Resize** – every image is converted to RGB and resized to **64 × 64** pixels, so each image becomes a `64 × 64 × 3` array (height × width × Red/Green/Blue channels). Traffic signs are simple, high-contrast symbols, so 64 × 64 keeps the shape, colour and symbol visible while keeping memory use and training time low. This size is a key part of handling the large dataset.

**2. Normalise** – pixel values naturally range from 0 to 255. They are divided by 255 so that they lie between **0 and 1**, which makes gradient-based training more stable and faster. To save memory, the arrays are stored as compact 8-bit integers (`uint8`, 4 times smaller than `float32`) and this division is performed by a **`Rescaling(1/255)` layer at the start of the CNN**. This is mathematically identical to normalising the arrays beforehand, and it also guarantees that new images given to the model at prediction time are normalised in exactly the same way as the training images.

The same `preprocess_image()` function is reused later when predicting a brand-new image, which guarantees that training images and new images are always treated in exactly the same way.

*Note:* some images are not perfectly square, so resizing to a square slightly changes the aspect ratio. Since this is applied consistently to every image (and to future images), the CNN simply learns from the resized versions.

In [ ]:
IMG_SIZE    = 64
INPUT_SHAPE = (IMG_SIZE, IMG_SIZE, 3)

def preprocess_image(path, img_size=IMG_SIZE):
    """Load an image, convert to RGB and resize it. Returns a uint8 array of shape (img_size, img_size, 3)."""
    with Image.open(path) as img:
        img = img.convert('RGB').resize((img_size, img_size), Image.LANCZOS)
        return np.asarray(img, dtype=np.uint8)

In [ ]:
# Visual check of the preprocessing on one image: original -> resized -> pixel-value distribution before/after normalisation
example_path = df.sample(1, random_state=SEED)['path'].values[0]
original = Image.open(example_path)
resized = preprocess_image(example_path)
normalised = resized.astype('float32') / 255.0      # the same division the Rescaling layer performs inside the CNN

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
axes[0].imshow(original);  axes[0].set_title(f'Original  {original.size[0]}x{original.size[1]}');  axes[0].axis('off')
axes[1].imshow(resized);   axes[1].set_title(f'Resized  {IMG_SIZE}x{IMG_SIZE}');                    axes[1].axis('off')
axes[2].hist(resized.ravel(), bins=50, color='steelblue')
axes[2].set_title(f'Pixel values BEFORE normalisation\n(min={resized.min()}, max={resized.max()})')
axes[3].hist(normalised.ravel(), bins=50, color='darkorange')
axes[3].set_title(f'Pixel values AFTER normalisation\n(min={normalised.min():.2f}, max={normalised.max():.2f})')
plt.tight_layout()
plt.show()

print('Array shape:', resized.shape, '| dtype before:', resized.dtype, '| dtype after:', normalised.dtype)

In [ ]:
# Preprocess the whole working set -> X (images, uint8) and y (integer labels: 0..NUM_CLASSES-1)
images = np.empty((len(df), IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)    # pre-allocated: no extra copy in memory
labels, paths = [], []

for k, (cls, p) in enumerate(tqdm(zip(df['class'], df['path']), total=len(df), desc='Preprocessing images')):
    images[k] = preprocess_image(p)
    labels.append(CLASS_TO_IDX[cls])
    paths.append(p)

X = images
y = np.array(labels)
paths = np.array(paths)
gc.collect()

print('X shape:', X.shape, '  (images, height, width, channels)')
print('y shape:', y.shape)
print('Pixel value range (before the Rescaling layer): {} to {}'.format(X.min(), X.max()))
print('Memory used by X: {:.0f} MB  (a float32 copy would need {:.0f} MB)'.format(X.nbytes / 1e6, X.nbytes * 4 / 1e6))

## **7. Train / Validation / Test Split**

The images are divided into three separate sets, each with a different purpose:

**1. Training set (70%)** – the images the CNN learns from.

**2. Validation set (15%)** – used *during development* to monitor training (early stopping) and to choose the best hyperparameters.

**3. Test set (15%)** – kept completely untouched until the very end and used only for the final, unbiased performance estimate on unseen images.

The split is **stratified**, which means every set keeps the same proportion of each sign as the working dataset. This is important here because the classes are not equally sized. **Class weights** are also computed so that mistakes on the smaller classes are penalised more heavily during training.

The file paths are split together with the images so that a test image can later be traced back to its original file.

In [ ]:
# Stratified 70 / 15 / 15 split (done on indices so that images, labels and file paths always stay aligned)
idx_all = np.arange(len(y))

idx_train, idx_temp = train_test_split(idx_all, test_size=0.30, stratify=y, random_state=SEED)
idx_val, idx_test   = train_test_split(idx_temp, test_size=0.50, stratify=y[idx_temp], random_state=SEED)

X_train, y_train = X[idx_train], y[idx_train]
X_val,   y_val   = X[idx_val],   y[idx_val]
X_test,  y_test  = X[idx_test],  y[idx_test]
paths_test = paths[idx_test]

# The full array is no longer needed -> free memory
del X, images, idx_temp
gc.collect()

print('Training set   :', X_train.shape, y_train.shape)
print('Validation set :', X_val.shape,   y_val.shape)
print('Test set       :', X_test.shape,  y_test.shape)

In [ ]:
# Check that the class proportions are preserved in every split
split_counts = pd.DataFrame({
    'Train':      pd.Series(y_train).map(dict(enumerate(CLASS_NAMES))).value_counts(),
    'Validation': pd.Series(y_val).map(dict(enumerate(CLASS_NAMES))).value_counts(),
    'Test':       pd.Series(y_test).map(dict(enumerate(CLASS_NAMES))).value_counts()
}).reindex(CLASS_NAMES)
split_counts['Total'] = split_counts.sum(axis=1)

display(split_counts.T)

split_counts[['Train', 'Validation', 'Test']].plot(kind='bar', stacked=True, figsize=(15, 5), colormap='viridis')
plt.title('Number of Images per Traffic Sign in Train / Validation / Test Sets')
plt.xlabel('Traffic Sign (Class)')
plt.ylabel('Number of Images')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Class weights (computed from the TRAINING labels only) to counter the class imbalance
class_weights_array = compute_class_weight(class_weight='balanced', classes=np.arange(NUM_CLASSES), y=y_train)
class_weights = {i: float(w) for i, w in enumerate(class_weights_array)}

print('Class weights (a value above 1 means the class is under-represented and gets extra importance):')
print({CLASS_NAMES[k]: round(float(v), 2) for k, v in class_weights.items()})

## **8. Data Augmentation**

A CNN can easily **memorise** the training photographs instead of learning the general appearance of each sign (overfitting). Data augmentation fights this by showing the network slightly modified versions of the training images at every epoch. The following random changes are used, which mimic real variation in how a camera sees a traffic sign from a moving vehicle:

1. **Rotation** of up to about ±11° (camera or sign slightly tilted).
2. **Translation** (shift) of up to 10% horizontally and vertically (sign not perfectly centred).
3. **Zoom** of up to 10% (sign closer to or further from the camera).
4. **Contrast change** of up to 20% (signs in bright sunlight, shade or at dusk).

**Horizontal flipping is deliberately NOT used.** Many traffic signs have a meaning that depends on direction – for example *"Turn left ahead"* and *"Turn right ahead"*, *"Keep left"* and *"Keep right"*, or *"Dangerous curve to the left"* and *"to the right"* are mirror images of each other. Flipping an image would turn one class into another and teach the model something that is incorrect.

The augmentation is built from Keras preprocessing layers and placed **at the start of the model**. These layers are active only during training and are automatically switched off during validation, testing and prediction, so the evaluation is always done on the original, unmodified images.

In [ ]:
# Augmentation block (active only during training)
data_augmentation = Sequential([
    RandomRotation(factor=0.03, fill_mode='nearest'),                              # +/- ~11 degrees
    RandomTranslation(height_factor=0.10, width_factor=0.10, fill_mode='nearest'), # +/- 10 % shift
    RandomZoom(height_factor=0.10, width_factor=0.10, fill_mode='nearest'),        # +/- 10 % zoom
    RandomContrast(factor=0.2),                                                    # +/- 20 % contrast
], name='data_augmentation')

# Visualise: one original training image and 7 random augmented versions of it
sample_img = X_train[0]
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes[0, 0].imshow(sample_img)
axes[0, 0].set_title('Original (' + label_of(y_train[0], 30).replace('\n', ' ') + ')', fontsize=10)
for ax in axes.ravel()[1:]:
    augmented = data_augmentation(np.expand_dims(sample_img, 0).astype('float32'), training=True)[0].numpy()
    ax.imshow(np.clip(augmented / 255.0, 0, 1))
    ax.set_title('Augmented')
for ax in axes.ravel():
    ax.axis('off')
plt.tight_layout()
plt.show()

## **9. Model Building & Training — Baseline CNN**

A **baseline** model is built first. It gives a reference score that the tuned model must beat, which shows whether hyperparameter tuning actually helps.

**How the CNN works (layer by layer):**

1. **Augmentation + Rescaling** – the random augmentation (training only) is applied, then pixel values are divided by 255 (normalisation, see Section 6).
2. **Convolution layers (`Conv2D`)** – slide small 3×3 filters over the image. The first layer detects simple features such as edges and lines; deeper layers combine them into shapes (circle, triangle, octagon) and finally symbols and numbers.
3. **Max pooling (`MaxPooling2D`)** – shrinks the image while keeping the strongest responses, making the model faster and less sensitive to the exact position of a feature.
4. **Flatten** – converts the extracted 2-D feature maps into a single vector.
5. **Dense layer + Dropout** – combines the features to decide between signs; Dropout randomly switches off some neurons during training to reduce overfitting.
6. **Softmax output layer** – produces one probability for each traffic sign class. The sign with the highest probability is the prediction.

The model is compiled with the **Adam** optimiser and **sparse categorical cross-entropy** loss (the labels are plain integers 0 to `NUM_CLASSES - 1` rather than one-hot vectors). Two callbacks are used: `EarlyStopping` stops training once the validation loss stops improving (and restores the best weights), and `ReduceLROnPlateau` lowers the learning rate when progress stalls.

In [ ]:
def build_baseline_cnn(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES):
    model = Sequential([
        Input(shape=input_shape),
        data_augmentation,
        Rescaling(1.0 / 255),                     # normalisation: 0-255  ->  0-1

        Conv2D(32, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),

        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),

        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),

        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ], name='baseline_cnn')

    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

baseline_model = build_baseline_cnn()
baseline_model.summary()

In [ ]:
# Training settings (reduce the epochs for a quicker run)
BASELINE_EPOCHS = 30
BATCH_SIZE      = 32

def get_callbacks(checkpoint_name, patience=5):
    """Fresh callbacks for every training run."""
    return [
        EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6),
        ModelCheckpoint(checkpoint_name, monitor='val_loss', save_best_only=True)
    ]

history_baseline = baseline_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=BASELINE_EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=get_callbacks('best_baseline_cnn.keras')
)

In [ ]:
# Helper: accuracy and loss curves. They show learning, convergence and overfitting
# (a growing gap between training and validation curves means overfitting)
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history.history['accuracy'], label='Training Accuracy')
    axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
    axes[0].set_title(f'{title} - Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].legend()

    axes[1].plot(history.history['loss'], label='Training Loss')
    axes[1].plot(history.history['val_loss'], label='Validation Loss')
    axes[1].set_title(f'{title} - Loss'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_history(history_baseline, 'Baseline CNN')

In [ ]:
# Helper used after every model is trained, so all models are evaluated in exactly the same way
def evaluate_model(model, X_data, y_data):
    y_prob = model.predict(X_data, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(y_data, y_pred, average='macro', zero_division=0)
    metrics = {
        'Accuracy': accuracy_score(y_data, y_pred),
        'Precision (macro)': precision,
        'Recall (macro)': recall,
        'F1-Score (macro)': f1
    }
    return metrics, y_pred, y_prob

baseline_val_metrics,  _, _ = evaluate_model(baseline_model, X_val, y_val)
baseline_test_metrics, y_pred_baseline, y_prob_baseline = evaluate_model(baseline_model, X_test, y_test)

print('Baseline CNN  - Validation accuracy: {:.4f}'.format(baseline_val_metrics['Accuracy']))
print('Baseline CNN  - Test accuracy      : {:.4f}'.format(baseline_test_metrics['Accuracy']))

## **10. Hyperparameter Tuning**

**Parameters vs. hyperparameters.** A CNN *learns* its weights and biases automatically during training. But some values must be chosen by us *before* training starts – these are **hyperparameters**. Instead of simply guessing them (for example "learning rate = 0.001"), different combinations are tried and the one that performs best on the **validation set** is selected. The test set is not used for this, so it stays a fair judge of the final model.

**Search space:**

| Hyperparameter | Values tried |
| --- | --- |
| Filters in the 1st convolution layer | 32, 64 |
| Filters in the 2nd convolution layer | 64, 128 |
| Dense layer units | 128, 256 |
| Dropout rate | 0.3, 0.4, 0.5 |
| Learning rate | 0.001, 0.0001 |
| Batch size | 16, 32 |

Trying every combination would be very slow (2 × 2 × 2 × 3 × 2 × 2 = 96 models), so **Keras Tuner's `RandomSearch`** is used: it randomly samples a fixed number of combinations (trials), trains each one briefly, and ranks them by **validation accuracy**. The same augmentation block and class weights are used in every trial, so the tuned model is compared with the baseline fairly.

*Tip:* increase `TUNER_TRIALS` to explore more combinations if time allows, or reduce it on a slow machine.

In [ ]:
TUNER_TRIALS = 8      # number of random hyperparameter combinations to try
TUNER_EPOCHS = 15     # maximum epochs per trial (EarlyStopping usually stops earlier)

class TrafficSignHyperModel(kt.HyperModel):
    """CNN whose important hyperparameters are chosen by Keras Tuner."""

    def build(self, hp):
        hp.Choice('batch_size', [16, 32])      # registered here so it is tuned/recorded; it is applied in fit() below
        model = Sequential([Input(shape=INPUT_SHAPE), data_augmentation, Rescaling(1.0 / 255)])

        model.add(Conv2D(hp.Choice('filters_1', [32, 64]), (3, 3), activation='relu'))
        model.add(MaxPooling2D((2, 2)))

        model.add(Conv2D(hp.Choice('filters_2', [64, 128]), (3, 3), activation='relu'))
        model.add(MaxPooling2D((2, 2)))

        model.add(Conv2D(128, (3, 3), activation='relu'))
        model.add(MaxPooling2D((2, 2)))

        model.add(Flatten())
        model.add(Dense(hp.Choice('dense_units', [128, 256]), activation='relu'))
        model.add(Dropout(hp.Float('dropout', 0.3, 0.5, step=0.1)))
        model.add(Dense(NUM_CLASSES, activation='softmax'))

        model.compile(optimizer=Adam(learning_rate=hp.Choice('learning_rate', [0.001, 0.0001])),
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])
        return model

    def fit(self, hp, model, *args, **kwargs):
        # batch size is also treated as a hyperparameter
        return model.fit(*args, batch_size=hp.get('batch_size'), **kwargs)

tuner = kt.RandomSearch(
    TrafficSignHyperModel(),
    objective='val_accuracy',
    max_trials=TUNER_TRIALS,
    seed=SEED,
    overwrite=True,
    directory='trafsign_tuning',
    project_name='trafsigndetc'
)

tuner.search_space_summary()

In [ ]:
# Run the search (each trial trains one randomly chosen combination and reports its validation accuracy)
tuner.search(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=TUNER_EPOCHS,
    class_weight=class_weights,
    callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)]
)

In [ ]:
# Safety check: make sure the trials really finished (Keras Tuner can skip failed trials without stopping)
n_completed = len([t for t in tuner.oracle.trials.values() if t.status == 'COMPLETED'])
print(f'Completed trials: {n_completed} / {TUNER_TRIALS}')
assert n_completed > 0, 'No tuning trial completed - scroll up to read the error message of the failed trials.'

### **10.1 Tuning Results & Best Hyperparameters**

Every trial is listed below, ranked by validation accuracy. The hyperparameters of the top-ranked trial are the **best hyperparameters**, and these are used to build the final model. They come directly from this notebook's own tuning run on the dataset, not from a pre-decided answer.

In [ ]:
# All trials, ranked by validation accuracy
trial_rows = []
for t in tuner.oracle.get_best_trials(num_trials=TUNER_TRIALS):
    row = dict(t.hyperparameters.values)
    row['val_accuracy'] = t.score
    trial_rows.append(row)

tuning_results = pd.DataFrame(trial_rows)
tuning_results.index = np.arange(1, len(tuning_results) + 1)
tuning_results.index.name = 'Rank'
tuning_results.round(4)

In [ ]:
# Best hyperparameters
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print('BEST HYPERPARAMETERS (selected on validation accuracy)')
print('-' * 55)
print('Filters (conv layer 1) :', best_hp.get('filters_1'))
print('Filters (conv layer 2) :', best_hp.get('filters_2'))
print('Dense layer units      :', best_hp.get('dense_units'))
print('Dropout rate           :', best_hp.get('dropout'))
print('Learning rate          :', best_hp.get('learning_rate'))
print('Batch size             :', best_hp.get('batch_size'))

## **11. Final CNN Model**

The tuner only trained each candidate for a short time. The final model is therefore built with the **best hyperparameters** and trained again from scratch for longer on the training set, still monitoring the validation set with `EarlyStopping` and `ReduceLROnPlateau`. Its accuracy and loss curves are plotted to check learning, convergence and overfitting.

In [ ]:
FINAL_EPOCHS     = 40
FINAL_BATCH_SIZE = best_hp.get('batch_size')

final_model = tuner.hypermodel.build(best_hp)
final_model.summary()

history_final = final_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=FINAL_EPOCHS,
    batch_size=FINAL_BATCH_SIZE,
    class_weight=class_weights,
    callbacks=get_callbacks('best_final_cnn.keras', patience=6)
)

In [ ]:
plot_history(history_final, 'Final (Tuned) CNN')

In [ ]:
final_val_metrics,  _, _ = evaluate_model(final_model, X_val, y_val)
final_test_metrics, y_pred_final, y_prob_final = evaluate_model(final_model, X_test, y_test)

print('Final CNN  - Validation accuracy: {:.4f}'.format(final_val_metrics['Accuracy']))
print('Final CNN  - Test accuracy      : {:.4f}'.format(final_test_metrics['Accuracy']))

## **12. Model Comparison Report**

The baseline CNN and the tuned final CNN are compared on both the validation and the test set using Accuracy and the macro-averaged Precision, Recall and F1-score. *Macro* averaging gives every sign equal importance regardless of how many images it has, which suits this imbalanced dataset.

The best model is **selected using the validation accuracy** (not the test accuracy), so the test set is never used to make a modelling decision. The test scores then give an honest estimate of how the selected model performs on unseen images.

In [ ]:
comparison = pd.DataFrame({
    'Baseline CNN - Validation': baseline_val_metrics,
    'Baseline CNN - Test':       baseline_test_metrics,
    'Final Tuned CNN - Validation': final_val_metrics,
    'Final Tuned CNN - Test':       final_test_metrics
}).T
comparison.round(4)

In [ ]:
# Visual comparison on the test set
test_comparison = comparison.loc[['Baseline CNN - Test', 'Final Tuned CNN - Test']]
test_comparison.index = ['Baseline CNN', 'Final Tuned CNN']

test_comparison.plot(kind='bar', figsize=(10, 5), colormap='viridis')
plt.title('Model Comparison on the Test Set')
plt.ylabel('Score')
plt.ylim(0, 1.05)
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Training curves of both models together
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, h in {'Baseline CNN': history_baseline, 'Final Tuned CNN': history_final}.items():
    axes[0].plot(h.history['val_accuracy'], label=f'{name} (validation)')
    axes[1].plot(h.history['val_loss'], label=f'{name} (validation)')
axes[0].set_title('Validation Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].set_title('Validation Loss');     axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Select the best model using VALIDATION accuracy
candidates = {
    'Baseline CNN':    (baseline_model, baseline_val_metrics, y_pred_baseline, y_prob_baseline),
    'Final Tuned CNN': (final_model,    final_val_metrics,    y_pred_final,    y_prob_final)
}
best_model_name = max(candidates, key=lambda k: candidates[k][1]['Accuracy'])
best_model, _, y_pred_best, y_prob_best = candidates[best_model_name]

print('Best model (highest validation accuracy):', best_model_name)
print('Its test accuracy: {:.4f}'.format(accuracy_score(y_test, y_pred_best)))

## **13. Best Model — Detailed Evaluation**

The selected model is evaluated in more depth on the **unseen test set**:

1. **Classification Report** – for each sign: *Precision* (of the images predicted as this sign, how many really were it?), *Recall* (of all real images of this sign, how many were found?), *F1-score* (the balance of the two) and *Support* (the number of test images).
2. **Confusion Matrix** – rows are the actual signs and columns are the predicted signs. Correct predictions lie on the diagonal; any value off the diagonal shows which signs are being confused with each other.
3. **Per-sign F1-score and most confused pairs** – to see exactly which signs the model finds easy or hard.
4. **Misclassified images** – to look at the actual mistakes.

In [ ]:
# Full classification report (class numbers with sign names)
report_names = [f'{c}: {SIGN_NAMES[c]}' if use_names else c for c in CLASS_NAMES]
print(f'Classification Report - {best_model_name} (Test Set)')
print(classification_report(y_test, y_pred_best, target_names=report_names, zero_division=0))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred_best, labels=np.arange(NUM_CLASSES))

plt.figure(figsize=(18, 15))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, annot_kws={'size': 6})
plt.xlabel('Predicted Sign (Class)', fontsize=13)
plt.ylabel('Actual Sign (Class)', fontsize=13)
plt.title(f'Confusion Matrix - {best_model_name} (Test Set)', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# Per-sign F1-score (which signs are easy / hard for the model?)
_, _, f1_per_class, support = precision_recall_fscore_support(y_test, y_pred_best, labels=np.arange(NUM_CLASSES), zero_division=0)
f1_series = pd.Series(f1_per_class, index=CLASS_NAMES).sort_values()

plt.figure(figsize=(15, 5))
sns.barplot(x=f1_series.index, y=f1_series.values, palette='viridis')
plt.title(f'F1-Score per Sign - {best_model_name} (sorted from weakest to strongest)')
plt.xlabel('Traffic Sign (Class)')
plt.ylabel('F1-Score')
plt.xticks(rotation=90)
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()

print('5 weakest signs  :', ', '.join(f'{c} - {SIGN_NAMES[c]} ({v:.2f})' for c, v in f1_series.head(5).items()))
print('5 strongest signs:', ', '.join(f'{c} - {SIGN_NAMES[c]} ({v:.2f})' for c, v in f1_series.tail(5).items()))

In [ ]:
# Most frequently confused pairs of signs (largest off-diagonal values of the confusion matrix)
confused = []
for actual in range(NUM_CLASSES):
    for predicted in range(NUM_CLASSES):
        if actual != predicted and cm[actual, predicted] > 0:
            confused.append({'Actual sign': f'{CLASS_NAMES[actual]}: {SIGN_NAMES[CLASS_NAMES[actual]]}',
                             'Predicted as': f'{CLASS_NAMES[predicted]}: {SIGN_NAMES[CLASS_NAMES[predicted]]}',
                             'Number of images': cm[actual, predicted]})

if confused:
    confused_df = pd.DataFrame(confused).sort_values('Number of images', ascending=False).head(10).reset_index(drop=True)
    display(confused_df)
else:
    print('No misclassified test images - the confusion matrix is perfectly diagonal.')

In [ ]:
# Look at some of the actual mistakes (original, un-resized photographs)
wrong_idx = np.where(y_pred_best != y_test)[0]
print(f'Misclassified test images: {len(wrong_idx)} out of {len(y_test)}')

if len(wrong_idx) > 0:
    show_idx = wrong_idx[:12]
    n_cols = 4
    n_rows = int(np.ceil(len(show_idx) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 4.6 * n_rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, i in zip(axes, show_idx):
        ax.imshow(Image.open(paths_test[i]))
        conf = y_prob_best[i, y_pred_best[i]] * 100
        ax.set_title(f'Actual: {label_of(y_test[i], 28)}\nPredicted: {label_of(y_pred_best[i], 28)}\n(confidence {conf:.0f}%)',
                     color='red', fontsize=9)
        ax.axis('off')
    for ax in axes[len(show_idx):]:
        ax.axis('off')
    plt.suptitle('Misclassified Test Images', fontsize=15)
    plt.tight_layout()
    plt.show()

## **14. Predicting the Traffic Sign of a New Image**

This section shows the complete application working end to end:

**New traffic sign image → Resize → CNN (normalise inside) → Probability for each sign → Highest probability → Predicted traffic sign**

The `predict_sign()` function reuses the *same* `preprocess_image()` function that was used for training, so a new image is treated exactly like the training images. It returns the predicted sign together with the model's **confidence** (the softmax probability) and the top-3 most likely signs. Note that confidence is the model's own estimate; it is **not a guarantee** that the prediction is correct.

In [ ]:
def predict_sign(image_path, model=None, top_k=3, show=True):
    """Predict the traffic sign shown in ONE image file. Returns (predicted_sign, confidence, top_k list)."""
    model = model if model is not None else best_model
    array = preprocess_image(image_path)                                    # resize (same as training); normalisation is inside the model
    probs = model.predict(np.expand_dims(array, axis=0), verbose=0)[0]      # add batch dimension -> probabilities
    top_idx = np.argsort(probs)[::-1][:top_k]
    top = [(label_of(i, 200), float(probs[i])) for i in top_idx]

    if show:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        axes[0].imshow(Image.open(image_path))
        axes[0].set_title(f'Predicted Sign: {label_of(top_idx[0], 30)}  ({top[0][1]*100:.1f}%)', fontsize=12)
        axes[0].axis('off')
        axes[1].barh([textwrap.fill(t[0], 30) for t in top][::-1], [t[1] * 100 for t in top][::-1], color='steelblue')
        axes[1].set_xlim(0, 100); axes[1].set_xlabel('Confidence (%)'); axes[1].set_title(f'Top-{top_k} predictions')
        plt.tight_layout()
        plt.show()
    return top[0][0], top[0][1], top

In [ ]:
# Demonstration on 4 random images from the unseen test set
random.seed(SEED)
for i in random.sample(range(len(paths_test)), 4):
    sign, conf, _ = predict_sign(paths_test[i])
    actual = label_of(y_test[i], 200)
    print(f'Actual sign: {actual}  |  Predicted sign: {sign}  |  Confidence: {conf*100:.1f}%  |  ',
          'CORRECT' if sign == actual else 'WRONG')
    print('-' * 80)

In [ ]:
# OPTIONAL: predict on your OWN image
# Option 1 (any environment): put the path of your image here, e.g. MY_IMAGE_PATH = 'my_sign.jpg'
MY_IMAGE_PATH = None
# Option 2 (Google Colab only): set to True to get an upload button
UPLOAD_OWN_IMAGE = False

if MY_IMAGE_PATH:
    predict_sign(MY_IMAGE_PATH)

if UPLOAD_OWN_IMAGE:
    try:
        from google.colab import files
        for fname in files.upload():
            predict_sign(fname)
    except ImportError:
        print('Upload is only available in Google Colab. Use MY_IMAGE_PATH instead.')

## **15. Save the Model**

The selected model is saved in the native Keras format (`.keras`) together with the class-name list, so that it can be loaded later (for example in a web app or mobile app) without retraining. The saved model is then re-loaded and its predictions are compared with the original model to confirm that it was saved correctly.

In [ ]:
MODEL_PATH = 'TrafficSign_CNN_Model.keras'
best_model.save(MODEL_PATH)

with open('TrafficSign_class_names.json', 'w') as f:
    json.dump({'class_folders': CLASS_NAMES, 'sign_names': SIGN_NAMES, 'img_size': IMG_SIZE}, f, indent=2)

# Verify: reload and compare predictions on 20 test images
reloaded_model = tf.keras.models.load_model(MODEL_PATH)
same = np.allclose(best_model.predict(X_test[:20], verbose=0),
                   reloaded_model.predict(X_test[:20], verbose=0), atol=1e-5)

print(f'Model saved as         : {MODEL_PATH}  ({os.path.getsize(MODEL_PATH)/1e6:.1f} MB)')
print('Class names saved as   : TrafficSign_class_names.json')
print('Reloaded model gives identical predictions:', same)

In [ ]:
# Final results summary (values are generated from THIS run, nothing is hard-coded)
final_acc = accuracy_score(y_test, y_pred_best)
final_prec, final_rec, final_f1, _ = precision_recall_fscore_support(y_test, y_pred_best, average='macro', zero_division=0)

print('=' * 64)
print('   PRAICP-1002 : TRAFFIC SIGN RECOGNITION - RESULTS')
print('=' * 64)
print(f'Number of traffic sign classes : {NUM_CLASSES}')
print(f'Images (train / val / test)    : {len(y_train)} / {len(y_val)} / {len(y_test)}')
print(f'Input size                     : {IMG_SIZE} x {IMG_SIZE} x 3')
print(f'Selected model                 : {best_model_name}')
print(f'Test accuracy                  : {final_acc*100:.2f} %')
print(f'Test precision (macro)         : {final_prec:.4f}')
print(f'Test recall (macro)            : {final_rec:.4f}')
print(f'Test F1-score (macro)          : {final_f1:.4f}')
print('=' * 64)

## **16. Business Insights**

**1. Road Safety and Driver Assistance:**<br>
The trained CNN turns a photograph of a traffic sign into a predicted sign class within a fraction of a second. Integrated into a driver-assistance system, such a model can alert drivers to speed limits, stop signs and hazard warnings they may have missed, helping to reduce accidents caused by missed or misread signs.<br><br>

**2. Foundation for Autonomous Vehicles:**<br>
Self-driving and semi-autonomous vehicles must read traffic signs to obey road rules. A reliable sign classifier is one of the building blocks of such systems, and the preprocessing and evaluation pipeline built here can be reused when more data or a detection stage is added.<br><br>

**3. Preprocessing Makes Real-World Use Possible:**<br>
The raw photographs came in many different resolutions and quality levels. Resizing and normalising every image (and applying exactly the same steps to new images at prediction time) is what allows one model to handle photos from different cameras consistently.<br><br>

**4. Confusion Analysis Shows Where to Improve:**<br>
The confusion matrix, the per-sign F1-scores and the misclassified-image gallery (Section 13) show which signs are confused with one another. Signs that look alike (for example different speed limits) are the best targets for collecting more varied training photographs, so that improvement effort can be focused instead of spread across the whole dataset. In a safety setting, confusing two speed limits can matter more than confusing two unrelated signs, so these errors deserve special attention.<br><br>

**5. Hyperparameter Tuning Adds Measurable Value:**<br>
The tuned model is compared against the untuned baseline on the same validation and test data (Section 12). Because the comparison is reported openly, the benefit of tuning can be judged from evidence instead of assumption.<br><br>

**6. Scope and Deployment Considerations:**<br>
The model *classifies* a sign that has already been cropped out of a photograph. A real system must also *detect* where signs are in a full road scene, work in rain, night and glare, and be validated thoroughly before it is relied on for any safety-critical decision.

## **17. Challenges Faced**

**1. Dataset Too Large to Load at Once:**<br>
The full dataset contains tens of thousands of images, which is too much to extract and hold in memory as arrays on normal hardware. This was solved by (a) reading the ZIP index without extracting it, (b) extracting only a fixed number of randomly chosen images per class (`MAX_IMAGES_PER_CLASS`), (c) resizing to 64×64 and (d) keeping the arrays as `uint8` and normalising inside the model. The cap can be raised or removed if more memory is available.<br><br>

**2. Images of Many Different Sizes and Quality:**<br>
The photographs vary widely in resolution and many are small, blurred, dark or bright. All were converted to a common 64×64 RGB format, and contrast augmentation was added to make the model robust to lighting changes.<br><br>

**3. Class Imbalance:**<br>
Some signs have many more images than others. This was handled by selecting images per class, stratified train/validation/test splitting and class weights during training, and the models are judged with macro-averaged metrics so that small classes count equally.<br><br>

**4. Visually Similar and Mirror-Image Signs:**<br>
Certain signs differ only by a small detail (such as the number on a speed-limit sign), and some signs are mirror images of each other (turn left / turn right). Horizontal flipping was therefore deliberately avoided in augmentation, and the confusion matrix and classification report are examined sign by sign instead of relying only on overall accuracy.<br><br>

**5. Risk of Overfitting:**<br>
A CNN can memorise the training photographs. Data augmentation (rotation, shift, zoom, contrast), dropout, early stopping and learning-rate reduction were all used to reduce overfitting.<br><br>

**6. Avoiding Information Leakage in Model Selection:**<br>
To keep the final score honest, hyperparameters and the best model were selected using only the validation set. The test set was used only once at the end for the final evaluation.<br><br>

**7. Compute Limits for Hyperparameter Search:**<br>
Searching many hyperparameter combinations can be slow. The number of tuner trials and epochs are configurable (`TUNER_TRIALS`, `TUNER_EPOCHS`) so the notebook can be run on modest hardware, and a GPU runtime (for example in Google Colab) is recommended.

## **18. Final Conclusion**

**1.Project Objective:**

The objective of this project was to build a machine learning model that recognises traffic signs from images, covering the two goals of the DataMites brief: image preprocessing and an ML model to predict the traffic sign.
<br><br>

**2.Exploratory Data Analysis (EDA):**

EDA showed a multi-class dataset of traffic sign photographs with unequal class sizes, many different image resolutions, and natural variation between images of the same sign (lighting, blur, size and angle) – which justified a memory-safe loading strategy, resizing, normalisation, stratified splitting, class weights and data augmentation.
<br><br>

**3.Modelling:**

A baseline CNN was built first, then Keras Tuner's random search was used to select the number of filters, dense units, dropout rate, learning rate and batch size using validation accuracy. A final CNN was trained with the best hyperparameters, and both models were compared using the same metrics.
<br><br>

**4.Outcome:**

The selected model's accuracy, macro precision/recall/F1-score, classification report and confusion matrix on the unseen test set are reported in Sections 12–13 and in the final results summary in Section 15. The model can predict the traffic sign of a new image with its confidence (Section 14) and has been saved for later use. Its remaining errors are concentrated in visually similar signs, and future work can include using the full dataset, transfer learning with pretrained networks, a detection stage to locate signs in full road scenes, and real-time camera recognition.

# **19. Project Links**

The following links provide access to the complete project resources, including the source code, dataset, and supporting documentation. Replace the placeholders below with your own repository / drive links before submission.

## **1. GitHub Repository**
Access the complete source code, Jupyter Notebook, README, and project files.

🔗 **GitHub Repository:**  *(paste your GitHub repository link here)*

---

## **2. Google Drive**
Access the PPT and REPORT.

🔗 **Google Drive:** *(paste your Google Drive folder link here)*